In [1]:
import os,sys
import pandas as pd
import math
import numpy as np

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
# import plotting packages
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.gridspec as gridspec

# Get the current directory
# os.getcwd()
# Get the parent directory (which is the 'main dir')
#os.path.dirname(current_dir)

code_dir=os.path.dirname(os.path.dirname(os.getcwd()))

# 1. Add the main directory to the import path
sys.path.append(code_dir)

# 2. Change the current working directory to the main directory
os.chdir(code_dir)

# RTM running script

In [3]:
# main_Surrogate_LUT_cases

# Table maker

In [36]:
# linux DeepL server
#!ls /home/dengnan/data/RTM/LUTcases/HG/
!ls /mnt/dengnan/LUTcases/HG/ | wc -l

1771


In [2]:
fdir = "/mnt/dengnan/LUTcases/HG/" 
Fls = os.listdir(fdir)
#targetregex = re.compile(r"Results_case2_COD=(\d+\.?\d*)_Tsurf=300_AOD=0\.0_COD=0\.0_th0=")
Fls = [f for f in Fls if f.startswith('Result')]
#Fls = [f for f in Fls if 'COD=20' in f and 'th0=30' in f]
Fls = np.sort(Fls)
print(len(Fls))
Fls[:5]

1771


array(['Results_case2_RH=0.1_Tsurf=233.0_AOD=0.1243_COD=15_th0=0.npy',
       'Results_case2_RH=0.1_Tsurf=233.0_AOD=0.1243_COD=15_th0=15.npy',
       'Results_case2_RH=0.1_Tsurf=233.0_AOD=0.1243_COD=15_th0=30.npy',
       'Results_case2_RH=0.1_Tsurf=233.0_AOD=0.1243_COD=15_th0=45.npy',
       'Results_case2_RH=0.1_Tsurf=233.0_AOD=0.1243_COD=15_th0=60.npy'],
      dtype='<U61')

### each channel srf dw os

In [3]:
from fun_nearealtime_RTM import FY4A_calinu, get_calibration_srf
import numpy as np
import pandas as pd
import os
file_dir = './FY4A_data/'
channels = ['C{:02d}'.format(c) for c in range(1, 6 + 1)]
nu0 = np.arange(2500, 35000, 3)  # Wavenumber grid
nu_channels = FY4A_calinu(nu0, channels, "./FY4A_data/", dnu=3)
# df = pd.DataFrame(columns=channels)

data = np.genfromtxt('data/profiles/ASTMG173.csv', delimiter=',', skip_header=2,  # in wavenumber basis
                    names=['wavelength', 'extraterrestrial', '37tilt', 'direct_circum'])
ref_lam = data['wavelength']  # nm avoid hearder 1
ref_E = data['extraterrestrial']
ref_E_nu = -ref_E * ref_lam ** 2 / 1e7  # W/[m2*nm-1] tp W/[m2*cm-1]

F_dw_os_srf_channel = []
for channel in channels:
    # load calibration data : Spectral Response Func
    srf, nu_channel = get_calibration_srf(channel, file_dir)
    F_dw_os_channel = -np.interp(-nu_channel, -1e7 / ref_lam, ref_E_nu)  # W/[m2*cm-1] to W/cm-1
    F_dw_os_SRF = np.multiply(F_dw_os_channel, srf)
    F_dw_os_srf_channel.append(np.trapz(F_dw_os_SRF, nu_channel))
    # nu_idx = np.nonzero(np.isin(nu_channels, nu_channel))[0]  # fixed 1 April.
    
    # # correct uw
    # uw_cor = np.multiply(uw[nu_idx], srf)
    # uw_channel = np.trapz(uw_cor,nu_channel)
    # df.loc[0, channel] = uw_channel
F_dw_os_srf_channel

[100.56360014402173,
 293.8703639771758,
 146.06104052297425,
 12.06884597258561,
 13.936208329862962,
 18.20438461023419]

In [4]:
np.save('./data/computed/F_dw_os_srf_channel.npy', F_dw_os_srf_channel)

### def


In [4]:
def fy4a_calibration_uw(uw):
    from fun_nearealtime_RTM import FY4A_calinu
    import numpy as np
    import pandas as pd
    import os
    file_dir='./FY4A_data/'
    channels = ['C{:02d}'.format(c) for c in range(1, 6 + 1)]
    nu0 = np.arange(2500, 35000, 3)  # Wavenumber grid
    nu_channels = FY4A_calinu(nu0, channels, "./FY4A_data/", dnu=3)
    df = pd.DataFrame(columns=channels)
    F_dw_os_srf_channel = [100.56360014402173,293.8703639771758,146.06104052297425,
                           12.06884597258561,13.936208329862962,18.20438461023419]

    for i, channel in enumerate(channels):
        srf, nu_channel = get_calibration_srf(channel, file_dir)
        nu_idx = np.nonzero(np.isin(nu0, nu_channel))[0]  # fixed 1 April.
        
        # correct uw
        uw_cor = np.multiply(uw[nu_idx], srf)
        uw_channel = np.trapz(uw_cor,nu_channel)
        # normalize uw_channel
        df.loc[0, channel] = uw_channel #/ F_dw_os_srf_channel[i]
    return df

In [21]:
ls

dengnan/  goldenzhaojian/  jingtao/  profchu/  SW_RTM/  user/  user1/  yuying/


In [ ]:
import numpy as np
import pandas as pd
import re
from tqdm import tqdm

# Step 1: Prepare your columns and lists
channels = ['C{:02d}'.format(c) for c in range(1, 7)]
Tsurf, RH, COD, th0 = [], [], [], [] # AOD,[]

# We'll build a list of dicts, each one a row
data_rows = []

nu0 = np.arange(2500, 35000, 3)

for fl in tqdm(Fls, desc="Processing Radiative Transfer Files"):
    # Load data
    results = dict(np.load(fdir+fl, allow_pickle=True)[0].items())
    #try:
    meta = {}    
    match = re.search(r'Tsurf=([\d.]+)', fl)
    if match: meta['Ta'] = float(match.group(1))

    match = re.search(r'RH=([\d.]+)', fl)
    if match: meta['rh'] = float(match.group(1))

    match = re.search(r'_COD=([\d.]+)', fl)
    if match: meta['COD'] = float(match.group(1))

    match = re.search(r'_th0=([\d.]+)', fl)
    if match: meta['th0'] = float(match.group(1))

    Fdw = results.get('F_dw')  # shape: (len(nu0), )
    meta['dsw'] = np.trapz(Fdw,nu0)
    DNI = results.get('F_dni')
    DHI = results.get('F_dhi')
    meta['dni'] = np.trapz(DNI, nu0)
    meta['dhi'] = np.trapz(DHI, nu0)

    uw = results.get('F_uw')
    df_uw_6channel = fy4a_calibration_uw(uw)
    # Save calibrated 6-channel values into meta
    for ch in df_uw_6channel.columns:
        meta[ch] = df_uw_6channel[ch].values[0]
    
    data_rows.append(meta)
# Step 2: Convert to pandas DataFrame
df = pd.DataFrame(data_rows)
print("\nProcessing Complete. DataFrame created.")

### Save the DataFrame to a CSV file

In [8]:
df.describe()

,Ta,rh,COD,th0,dsw,dni,dhi,C01,C02,C03,C04,C05,C06
count,4200.000000,4200.000000,4200.000000,4200.000000,4200.000000,4200.000000,4200.000000,4200.000000,4200.000000,4200.000000,4200.000000,4200.000000,4200.000000
mean,288.000000,0.550000,2.243333,30.000000,693.683678,273.446209,420.237469,12.373771,17.750823,8.717674,0.009244,2.734393,0.609898
std,10.001191,0.287262,2.968482,21.215729,241.041006,248.980606,189.745889,5.459830,9.377276,2.406898,0.021117,0.673943,0.148958
min,273.000000,0.100000,0.000000,0.000000,178.630593,0.203471,129.223464,6.837568,7.267078,4.376251,0.000995,1.412268,0.288493
25%,278.000000,0.300000,0.227500,15.000000,481.670138,29.925123,266.353122,8.630519,11.393284,7.091137,0.001964,2.256755,0.516865
50%,288.000000,0.550000,0.815000,30.000000,711.932591,212.243348,376.791751,9.712171,13.687193,8.642005,0.003071,2.793544,0.619201
75%,298.000000,0.800000,2.877500,45.000000,912.874309,471.069676,570.409849,14.177186,20.758461,9.540912,0.006104,3.024452,0.676841
max,303.000000,1.000000,10.000000,60.000000,1084.362190,870.848725,887.630828,29.590277,46.456967,15.889323,0.278046,4.569512,0.981711


In [9]:
df['dni'] = df['dni']/np.cos(np.deg2rad(df['th0']))

In [13]:
df.to_hdf('GOES_data/yuying_cases/SWRTM_case2_54layers_dnu=3_AOD=0.1243_yuyingCases4200.h5', key='data', mode='w')

In [ ]:
import pandas as pd

# The path to your HDF5 file
file= 'GOES_data/yuying_cases/SWRTM_case2_54layers_dnu=3_AOD=0.1243_yuyingCases7200.h5'
df7200 = pd.read_hdf(file, key='data')

df_combined = pd.concat([df, df7200], ignore_index=True)
    
print("\n✅ DataFrames combined successfully.")
print(f"Original df rows: {len(df)}")
print(f"df7200 rows: {len(df7200)}")
print(f"Combined df_combined rows: {len(df_combined)}")
print(f"Combined DataFrame shape: {df_combined.shape}")



✅ DataFrames combined successfully.
Original df rows: 4200
df7200 rows: 7200
Combined df_combined rows: 11400
Combined DataFrame shape: (11400, 13)


In [19]:
df_combined.to_hdf('GOES_data/yuying_cases/SWRTM_case2_54layers_dnu=3_AOD=0.1243_yuyingCases.h5', key='data', mode='w')

In [20]:
df_combined.describe()

,Ta,rh,COD,th0,dsw,dni,dhi,C01,C02,C03,C04,C05,C06
count,11400.000000,11400.000000,11400.000000,11400.000000,11400.000000,11400.000000,11400.000000,11400.000000,11400.000000,11400.000000,11400.000000,11400.000000,11400.000000
mean,258.000000,0.550000,2.243333,30.000000,736.767993,346.780438,444.435377,12.275012,17.693917,8.677902,0.286520,2.723279,0.617149
std,27.387329,0.287241,2.968259,21.214134,256.232657,287.669677,202.767732,5.427836,9.334839,2.393866,0.350660,0.673553,0.149405
min,213.000000,0.100000,0.000000,0.000000,178.630593,0.203471,129.223464,6.832717,7.267078,4.345912,0.000995,1.401151,0.288493
25%,233.000000,0.300000,0.227500,15.000000,513.967142,41.265748,279.216937,8.540242,11.484848,7.074305,0.004732,2.264506,0.530272
50%,258.000000,0.550000,0.815000,30.000000,751.801557,334.438331,392.672427,9.680281,13.603467,8.637425,0.098395,2.770524,0.627542
75%,283.000000,0.800000,2.877500,45.000000,954.345229,612.796348,604.271294,14.121270,20.579261,9.475528,0.520591,2.998217,0.688593
max,303.000000,1.000000,10.000000,60.000000,1138.591306,914.537632,934.209911,29.590277,46.456967,15.889323,1.699142,4.569512,0.981711
